# L07 · Why OPD succeeds or fails

## Goal

**Estimated time:** 40 min · **Path:** full

- measure support overlap
- interpret failure signals
- explain the vOPD baseline

### Current position: L06 → **L07** → L08

```text
Prompt/Data -> state source -> ... -> L07 -> ... -> fair evaluation
```

Alt text: The course map highlights L07 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L07"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L07', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

A stronger teacher does not guarantee successful OPD. Measure top-k support on student-visited states and ask whether the teacher supplies genuinely useful novelty.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

Separate OPD failures into at least three layers: (1) the teacher is wrong on that state, (2) teacher/student support differs so much that useful tokens are never sampled, or (3) optimization fails through estimator variance, learning rate, or stale rollouts. One loss curve cannot distinguish them.

Read top-k overlap, entropy gap, both KL directions, and intervention rate together. vOPD uses a detached top-k KL baseline to reduce sampled-estimator variance. A baseline must not alter the expected gradient and is not the optimized target.

### Production implementation: why this design

Support diagnostics compute top-k set overlap and entropy gap only on response targets. The vOPD baseline is detached; only sampled-token log-probabilities carry gradient. Thresholds are alert criteria across comparable runs, not performance guarantees.

Production code: [`support.py`](../../src/opd_study/diagnostics/support.py), [`vopd.py`](../../src/opd_study/algorithms/vopd.py).

In [2]:
import inspect
from opd_study.diagnostics import support_diagnostics
from opd_study.algorithms import vopd_loss

objects_to_show = (support_diagnostics, vopd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.diagnostics.support.support_diagnostics
def support_diagnostics(
    student_logits: Tensor,
    teacher_logits: Tensor,
    mask: Tensor,
    *,
    top_k: int,
) -> SupportDiagnostics:
    """Measure top-k overlap and alignment only on explicitly selected states."""

    if student_logits.shape != teacher_logits.shape:
        raise ValueError("student and teacher logits must match")
    if mask.shape != student_logits.shape[:-1]:
        raise ValueError("mask must match non-vocabulary logits dimensions")
    vocabulary_size = student_logits.shape[-1]
    if not 1 <= top_k <= vocabulary_size:
        raise ValueError(f"top_k must be within [1, {vocabulary_size}]")
    selected_mask = mask.bool().reshape(-1)
    if not selected_mask.any().item():
        raise ValueError("support diagnostics require at least one selected token")
    student_log = torch.log_softmax(student_logits.float(), dim=-1).reshape(
        -1, vocabulary_size
    )[selected_mask]
    teacher_log = 

### Alternatives and trade-offs

Possible overlap remedies include a different teacher, temperature, off/on-policy mixture, curriculum, or multiple samples. Classify the failure layer first, then ablate one change at a time instead of applying every remedy to one bad metric.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L07's output? Write one sentence, then run.

In [3]:
from opd_study.diagnostics import support_diagnostics

student = torch.tensor([[[4., 3., 1., 0.], [0., 1., 3., 4.]]])
compatible = student + torch.tensor([[[.1, 0., 0., 0.], [0., 0., 0., .1]]])
incompatible = student.flip(-1)
mask = torch.tensor([[True, True]])
for name, teacher in (("compatible", compatible), ("incompatible", incompatible)):
    result = support_diagnostics(student, teacher, mask, top_k=2)
    print(name, {"overlap": result.overlap_ratio, "entropy_gap": result.absolute_entropy_gap})

compatible {'overlap': 1.0, 'entropy_gap': 0.029091089963912964}
incompatible {'overlap': 0.0, 'entropy_gap': 0.0}


In [4]:
from opd_study.algorithms import vopd_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.types import TeacherSignals

tokenizer = CharacterTokenizer(); row = generate_tiny_arithmetic(train_rows=1, validation_rows=1, test_rows=1).train
batch = collate_examples(row, tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student_logits = torch.randn(shape, requires_grad=True); teacher_logits = torch.randn(shape)
vopd = vopd_loss(student_logits, batch, TeacherSignals(logits=teacher_logits), baseline_top_k=8)
print("vOPD reward/advantage:", vopd.metrics["vopd/mean_reward"], vopd.metrics["vopd/mean_advantage"])
print("The top-k KL is a detached baseline, not the optimized target.")

vOPD reward/advantage: -0.010935024358332157 0.4987005889415741
The top-k KL is a detached baseline, not the optimized target.


## Checks

In [5]:
good = support_diagnostics(student, compatible, mask, top_k=2)
bad = support_diagnostics(student, incompatible, mask, top_k=2)
assert good.overlap_ratio > bad.overlap_ratio
assert vopd.loss.requires_grad
print("check passed: overlap diagnoses state compatibility; vOPD keeps sampled-token gradients")

check passed: overlap diagnoses state compatibility; vOPD keeps sampled-token gradients


**Exercise (8 min):** create logits with low overlap but a small entropy gap. Explain why that alone cannot establish a bad teacher.

<details><summary>Check</summary>Support ranking and sharpness are different axes; correctness/environment-success evidence is still needed.</details>

## My recurring mistakes

### M1 — Declaring a bad teacher from low overlap

- Wrong: different top-k sets mean the teacher is wrong.
- Why: the student may not yet visit a useful teacher mode.
- Fix: combine correctness, entropy, KL, and environment success.
- Related check: `test_identical_support_is_perfectly_aligned`

### M2 — Differentiating the vOPD baseline

- Wrong: optimize the baseline as a target.
- Why: it can alter the expected score-function gradient.
- Fix: detach the baseline; differentiate sampled log-probability only.
- Related check: `test_vopd_is_zero_when_teacher_equals_student`

## 60-second summary

1. measure support overlap
2. interpret failure signals
3. explain the vOPD baseline

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`rethinking_opd`](https://arxiv.org/abs/2604.13016v2) · `2604.13016v2` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`vopd`](https://arxiv.org/abs/2605.07865v1) · `2605.07865v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)